In [ ]:
# 导入必要的库
import geopandas as gpd
import pandas as pd
import os
from pathlib import Path

# 设置文件路径
folder_path = r'E:\new-york-251023-free'  # 修正路径
desktop_path = os.path.join(os.path.expanduser('~'), 'Desktop')
output_file = os.path.join(desktop_path, 'NY_POI_class.csv')

print("=" * 60)
print("纽约POI数据fclass字段统计分析")
print("=" * 60)
print(f"文件夹路径: {folder_path}")
print(f"输出文件: {output_file}")

# 检查文件夹是否存在
if not os.path.exists(folder_path):
    print(f"❌ 错误: 文件夹 {folder_path} 不存在")
    # 尝试备用路径
    backup_path = r'E:\new-york-251023-free'
    if os.path.exists(backup_path):
        folder_path = backup_path
        print(f"✅ 使用备用路径: {folder_path}")
    else:
        print("❌ 备用路径也不存在，请检查路径设置")
else:
    print("✅ 文件夹路径验证通过")

# 获取所有shp文件
try:
    shp_files = list(Path(folder_path).glob("*.shp"))
    
    if len(shp_files) == 0:
        # 尝试递归搜索
        shp_files = list(Path(folder_path).rglob("*.shp"))
        if len(shp_files) > 0:
            print(f"🔍 在子文件夹中找到 {len(shp_files)} 个SHP文件")
        else:
            print("❌ 错误: 在指定文件夹及子文件夹中未找到SHP文件")
    else:
        print(f"✅ 找到 {len(shp_files)} 个SHP文件")
        
    # 显示找到的文件
    if len(shp_files) > 0:
        print("\n📂 找到的SHP文件列表:")
        for i, file in enumerate(shp_files, 1):
            relative_path = file.relative_to(Path(folder_path))
            print(f"  {i:2d}. {relative_path}")
    
except Exception as e:
    print(f"❌ 搜索SHP文件时出错: {str(e)}")
    shp_files = []

# 处理SHP文件
if len(shp_files) > 0:
    print(f"\n" + "=" * 60)
    print("开始处理SHP文件...")
    print("=" * 60)
    
    # 创建一个空的DataFrame来存储结果
    all_stats = []
    processed_files = 0
    error_files = 0
    
    # 遍历每个shp文件
    for i, shp_file in enumerate(shp_files, 1):
        try:
            print(f"\n[{i}/{len(shp_files)}] 正在处理: {shp_file.name}")
            
            # 读取shp文件
            gdf = gpd.read_file(shp_file)
            print(f"  ✓ 成功读取 - 记录数: {len(gdf):,}")
            
            # 检查是否存在fclass字段
            if 'fclass' in gdf.columns:
                # 统计fclass字段的类别和出现次数
                fclass_counts = gdf['fclass'].value_counts().reset_index()
                fclass_counts.columns = ['fclass', 'count']
                
                # 添加文件名列
                fclass_counts['filename'] = shp_file.name
                
                # 添加到总结果中
                all_stats.append(fclass_counts)
                
                print(f"  ✓ 发现 fclass 字段 - 类别数: {len(fclass_counts)}")
                processed_files += 1
            else:
                print(f"  ⚠ 警告: 文件中没有fclass字段")
                # 创建一个包含文件信息的记录
                empty_df = pd.DataFrame({
                    'fclass': ['NO_FCLASS_FIELD'],
                    'count': [0],
                    'filename': [shp_file.name]
                })
                all_stats.append(empty_df)
                processed_files += 1
                
        except Exception as e:
            print(f"  ❌ 错误: 处理文件时出错: {str(e)}")
            # 创建一个包含错误信息的记录
            error_df = pd.DataFrame({
                'fclass': ['ERROR_PROCESSING_FILE'],
                'count': [0],
                'filename': [shp_file.name]
            })
            all_stats.append(error_df)
            error_files += 1
    
    # 处理结果
    if all_stats:
        # 合并所有统计结果
        final_df = pd.concat(all_stats, ignore_index=True)
        
        # 重新排列列的顺序
        final_df = final_df[['filename', 'fclass', 'count']]
        
        # 按文件名和fclass排序
        final_df = final_df.sort_values(['filename', 'fclass'])
        
        # 保存到CSV文件
        try:
            final_df.to_csv(output_file, index=False, encoding='utf-8-sig')
            print(f"\n" + "=" * 60)
            print("✅ 处理完成!")
            print("=" * 60)
            print(f"📊 统计结果已保存到: {output_file}")
            print(f"📁 总文件数: {len(shp_files)}")
            print(f"✅ 成功处理: {processed_files}")
            print(f"❌ 处理失败: {error_files}")
            print(f"📋 总记录数: {len(final_df):,}")
            
            # 显示统计摘要
            valid_records = final_df[~final_df['fclass'].isin(['NO_FCLASS_FIELD', 'ERROR_PROCESSING_FILE'])]
            if len(valid_records) > 0:
                total_poi_count = valid_records['count'].sum()
                unique_categories = valid_records['fclass'].nunique()
                print(f"📍 POI记录总数: {total_poi_count:,}")
                print(f"🏷️  不同类别总数: {unique_categories}")
                
                # 显示最常见的类别
                top_categories = (valid_records.groupby('fclass')['count']
                               .sum()
                               .sort_values(ascending=False)
                               .head(10))
                print(f"\n🏆 最常见的10个fclass类别:")
                for i, (category, count) in enumerate(top_categories.items(), 1):
                    print(f"  {i:2d}. {category}: {count:,}")
            
            # 显示前几行结果
            print(f"\n📋 结果预览 (前15行):")
            print(final_df.head(15).to_string(index=False))
            
            # 按文件统计
            file_stats = final_df.groupby('filename').agg({
                'count': 'sum',
                'fclass': 'count'
            }).rename(columns={'count': 'total_records', 'fclass': 'categories'})
            
            print(f"\n📊 各文件统计:")
            for idx, row in file_stats.iterrows():
                if row['total_records'] > 0:
                    print(f"  📄 {idx}: {row['total_records']:,} 条记录, {row['categories']} 个类别")
            
        except Exception as e:
            print(f"❌ 保存文件时出错: {str(e)}")
    else:
        print("❌ 没有成功处理任何文件")
else:
    print("❌ 未找到可处理的SHP文件")

print(f"\n" + "=" * 60)
print("程序执行完毕")
print("=" * 60)
